# Phase 5 — Explainability

[`walkthrough/phase5.md`](../walkthrough/phase5.md). Both tuned models from Phase 4
(`results/model_final.json`) are explained on the held-out PSUs: the headline (deployable)
model, and the full model whose spending shares the Phase 1 leakage check flagged. SHAP values
come from `shap.TreeExplainer`, are summed within feature families, and are cross-checked by
grouped permutation importance.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline

from savings_goal import config as C
from savings_goal import viz
from savings_goal.evaluation import explain
from savings_goal.features.engineer import spec_from_frame
from savings_goal.io import load_features, write_result
from savings_goal.models.pipeline import load_model_final, logistic, preprocessor, xgb_pipeline

viz.use()
feats = load_features()
spec = spec_from_frame(feats)
final = load_model_final()
train = feats[~feats["Is_Test"]].reset_index(drop=True)
test = feats[feats["Is_Test"]].reset_index(drop=True)
SETS = {"deployable": spec.deployable_numeric, "full": spec.numeric}

fitted, Xt, names = {}, {}, {}
for name, num in SETS.items():
    cols = num + C.CATEGORICALS
    m = xgb_pipeline(num, params=final[name]["best_params"]).fit(train[cols], train["Goal_Met"])
    fitted[name] = m
    names[name] = list(m[:-1].get_feature_names_out())
    Xt[name] = pd.DataFrame(m[:-1].transform(test[cols]), columns=names[name])
    auc = roc_auc_score(test["Goal_Met"], m.predict_proba(test[cols])[:, 1])
    print(f"{name:10s} {len(names[name])} encoded columns; held-out ROC-AUC {auc:.4f}")

deployable 35 encoded columns; held-out ROC-AUC 0.8860


full       51 encoded columns; held-out ROC-AUC 0.9310


## The identification problem: why no coefficient on the shares can be read

In [2]:
num = spec.numeric


def lin(drop_reference):
    return lambda: make_pipeline(preprocessor(num, linear=True, drop_reference=drop_reference), logistic(C=100.0))


X_lin = train[num + C.CATEGORICALS]
all11 = explain.coefficient_stability(lin(False), X_lin, train["Goal_Met"])
ref = explain.coefficient_stability(lin(True), X_lin, train["Goal_Met"])
shares = [c for c in C.SHARE_COLS if c != "Groceries_Share"]
stab = pd.DataFrame({
    "sd (all 11 shares)": all11.loc[shares, "sd"], "sd (Groceries dropped)": ref.loc[shares, "sd"],
    "flips sign (all 11)": all11.loc[shares, "flips_sign"], "flips sign (dropped)": ref.loc[shares, "flips_sign"],
})
stab["sd ratio (all / dropped)"] = stab["sd (all 11 shares)"] / stab["sd (Groceries dropped)"]
stab["zero rate"] = (train[shares] == 0).mean()
print("Coefficient stability over 5 disjoint training subsamples, every share (not just the median):")
print(stab.sort_values("sd ratio (all / dropped)").round(3).to_string())
worse = stab[stab["sd ratio (all / dropped)"] < 1]
print(f"\nreference-part fix: median sd ratio {stab['sd ratio (all / dropped)'].median():.2f}x; "
      f"{len(worse)} share(s) get LESS stable: "
      + ", ".join(f"{i} ({1 / r:.1f}x worse, zero in {z:.0%})"
                  for i, r, z in zip(worse.index, worse["sd ratio (all / dropped)"], worse["zero rate"])))

Coefficient stability over 5 disjoint training subsamples, every share (not just the median):
                         sd (all 11 shares)  sd (Groceries dropped)  flips sign (all 11)  flips sign (dropped)  sd ratio (all / dropped)  zero rate
Healthcare_Share                      0.051                   0.070                False                 False                     0.724      0.193
Miscellaneous_Share                   0.043                   0.049                False                 False                     0.876      0.001
Utilities_Share                       0.028                   0.031                False                 False                     0.879      0.002
Transport_Share                       0.055                   0.062                False                 False                     0.885      0.114
Insurance_Share                       0.041                   0.044                 True                 False                     0.951      0.737
Clothing_Footwear_

## Q1 — Which features matter globally? Per column, per family, and by permutation

In [3]:
shap_vals, family_tbl, perm_tbl, per_col = {}, {}, {}, {}
for name in SETS:
    values, base, err = explain.tree_shap(fitted[name][-1], Xt[name])
    shap_vals[name] = values
    print(f"{name}: additivity check, max |sum(SHAP) + base - margin| = {err:.2e}")
    per_col[name] = pd.Series(np.abs(values).mean(axis=0), index=names[name]).sort_values(ascending=False)
    family_tbl[name] = explain.grouped_shap(values, names[name], explain.feature_families(names[name]))
    perm_tbl[name] = explain.grouped_permutation_importance(
        fitted[name], test[SETS[name] + C.CATEGORICALS], test["Goal_Met"],
        explain.raw_families(SETS[name] + C.CATEGORICALS), roc_auc_score, n_repeats=5)

for name in SETS:
    print(f"\n=== {name} model: top 10 columns by mean |SHAP| ===")
    print(per_col[name].head(10).round(4).to_string())
    both = family_tbl[name][["grouped mean |SHAP|", "share of grouped total"]].join(
        perm_tbl[name][["importance", "share"]].rename(columns={"importance": "ROC-AUC drop (permuted)",
                                                                "share": "share of permutation total"}))
    print(f"\n=== {name} model: feature families (SHAP summed within family vs permutation) ===")
    print(both.round(4).to_string())

deployable: additivity check, max |sum(SHAP) + base - margin| = 5.87e-06


full: additivity check, max |sum(SHAP) + base - margin| = 1.76e-05



=== deployable model: top 10 columns by mean |SHAP| ===
Log_Income                          1.9619
Household_Size                      0.4855
Debt_To_Income                      0.2875
Max_Adult_Education                 0.2699
Occupation_Non_Ag_Labour            0.1522
Dependency_Ratio                    0.1211
Area_Type_Less_Developed_Village    0.1211
Head_Age                            0.0908
Area_Type_Metro_Urban               0.0730
Caste_Group_Scheduled_Tribe         0.0677

=== deployable model: feature families (SHAP summed within family vs permutation) ===
                           grouped mean |SHAP|  share of grouped total  ROC-AUC drop (permuted)  share of permutation total
family                                                                                                                     
Income                                  1.9619                  0.5823                   0.3855                      0.8152
Household size family                   0.4785        

In [4]:
for name in SETS:
    fig = plt.figure(figsize=(9, 6.5))
    shap.summary_plot(shap_vals[name], Xt[name].to_numpy(), feature_names=names[name], max_display=15,
                      show=False, cmap=viz.SEQUENTIAL, color_bar_label="feature value (low to high)")
    ax = plt.gca()
    viz.strip(ax, "x")
    ax.axvline(0, color=viz.AXIS, lw=0.8, zorder=0)
    ax.set_title(f"SHAP values, tuned XGBoost on the {name} features (held-out PSUs)")
    plt.savefig(C.RESULTS / ("shap_summary.png" if name == "deployable" else "shap_summary_full.png"))
    plt.show()

## Q2 — Interactions, decomposed over the whole held-out set

In [5]:
inter = {}
for name in SETS:
    totals, pairs = explain.interaction_decomposition(fitted[name][-1], Xt[name], names[name])
    inter[name] = (totals, pairs)
    print(f"{name}: main effects {totals['main effects']:.4f}, interactions {totals['interactions']:.4f} "
          f"-> interaction share {totals['interaction share']:.1%} over {int(totals['n rows']):,} rows")
    print(pairs.head(6).round(4).to_string(index=False))
    print()

deployable: main effects 3.8495, interactions 1.8157 -> interaction share 32.1% over 8,299 rows
                         level_0                  level_1  mean |interaction SHAP|
                  Household_Size               Log_Income                   0.2425
                      Log_Income      Max_Adult_Education                   0.1313
                  Debt_To_Income               Log_Income                   0.0598
Area_Type_Less_Developed_Village               Log_Income                   0.0474
                      Log_Income Occupation_Non_Ag_Labour                   0.0446
                        Head_Age               Log_Income                   0.0419



full: main effects 6.9747, interactions 5.8772 -> interaction share 45.7% over 8,299 rows
        level_0             level_1  mean |interaction SHAP|
Groceries_Share          Log_Income                   0.3084
 Household_Size          Log_Income                   0.1774
     Log_Income     Utilities_Share                   0.1624
Groceries_Share     Utilities_Share                   0.0866
     Log_Income     Transport_Share                   0.0865
     Log_Income Max_Adult_Education                   0.0789



## The grocery-share "reversal" is the reconstruction route

Conditional on income and household size, a higher food share means a *smaller* total
budget (total = food / share, and food tracks size), so it predicts being on track. The
effect reconstructs the label and says nothing about spending behaviour.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.subplots_adjust(wspace=0.3)
sv = pd.DataFrame(shap_vals["deployable"], columns=names["deployable"])
ax = axes[0]
ax.scatter(test["Log_Income"], sv["Log_Income"], s=3, color=viz.CONTEXT, alpha=0.5, linewidths=0, zorder=1)
bins = np.linspace(test["Log_Income"].quantile(0.01), test["Log_Income"].quantile(0.99), 16)
AREAS = ["Metro_Urban", "Other_Urban", "Developed_Village", "Less_Developed_Village"]
for colour, area in zip(viz.CATEGORICAL, AREAS):
    m = (test["Area_Type"] == area).to_numpy()
    cut = pd.cut(test.loc[m, "Log_Income"], bins)
    med = sv.loc[m, "Log_Income"].groupby(cut.to_numpy(), observed=True).median()
    ax.plot([iv.mid for iv in med.index], med.values, color=colour, lw=2, label=area.replace("_", " "), zorder=3)
ax.axhline(0, color=viz.AXIS, lw=0.8, zorder=2)
viz.strip(ax, "y")
ax.set_xlabel("log annual income (Rs)")
ax.set_ylabel("SHAP value of Log_Income (log-odds)")
viz.legend(ax, loc="upper left", title="binned median by area type", title_fontsize=8)
fam = family_tbl["deployable"]
ax.set_title(f"Headline model: income is {fam.loc['Income', 'share of grouped total']:.0%} of grouped "
             "attribution,\nand its effect barely differs by area type")

sv_full = pd.DataFrame(shap_vals["full"], columns=names["full"])
ax = axes[1]
implied_total = test["Household_Size"] / test["Groceries_Share"].where(test["Groceries_Share"] > 0)
ratio = np.log(implied_total / np.exp(test["Log_Income"]))
sc = ax.scatter(test["Groceries_Share"], sv_full["Groceries_Share"], s=3, linewidths=0, alpha=0.6,
                c=ratio.clip(ratio.quantile(0.02), ratio.quantile(0.98)), cmap=viz.SEQUENTIAL)
ax.axhline(0, color=viz.AXIS, lw=0.8)
viz.strip(ax, "y")
ax.set_xlabel("food share of total expenditure")
ax.set_ylabel("SHAP value of Groceries_Share (full model)")
rho = pd.Series(sv_full["Groceries_Share"]).corr(-ratio, method="spearman")
ax.set_title(f"Food-share SHAP rises as implied spend/income falls (Spearman {rho:.2f}),\n"
             "consistent with the size x food-share reconstruction")
cb = fig.colorbar(sc, ax=ax, label="log(size / food share / income)")
cb.outline.set_visible(False)
plt.savefig(C.RESULTS / "shap_dependence.png")
plt.show()

## Q3 — Individual predictions in plain language (contributions only; no household's raw values are printed)

In [7]:
p_all = fitted["deployable"].predict_proba(test[SETS["deployable"] + C.CATEGORICALS])[:, 1]


def explain_row(i, top=5):
    contrib = sv.iloc[i].sort_values(key=np.abs, ascending=False).head(top)
    print(f"predicted P(on track) = {p_all[i]:.1%}; actual: "
          f"{'on track' if test['Goal_Met'].iloc[i] == 1 else 'at risk'}")
    for feat, val in contrib.items():
        print(f"    {val:+.3f}  {'toward on track' if val > 0 else 'toward at risk':16s} {feat}")
    print()


for label, i in [("most confident on track", int(np.argmax(p_all))),
                 ("most confident at risk", int(np.argmin(p_all))),
                 ("borderline", int(np.argmin(np.abs(p_all - 0.5))))]:
    print(label.upper())
    explain_row(i)

MOST CONFIDENT ON TRACK
predicted P(on track) = 99.3%; actual: on track
    +4.549  toward on track  Log_Income
    +0.498  toward on track  Max_Adult_Education
    +0.271  toward on track  Dependency_Ratio
    +0.198  toward on track  Area_Type_Less_Developed_Village
    +0.179  toward on track  Debt_To_Income

MOST CONFIDENT AT RISK
predicted P(on track) = 0.0%; actual: at risk
    -5.407  toward at risk   Log_Income
    -0.914  toward at risk   Debt_To_Income
    -0.840  toward at risk   Household_Size
    -0.262  toward at risk   Max_Adult_Education
    -0.156  toward at risk   Head_Age

BORDERLINE
predicted P(on track) = 50.0%; actual: on track
    +1.228  toward on track  Household_Size
    -0.463  toward at risk   Max_Adult_Education
    -0.305  toward at risk   Occupation_No_Regular_Worker
    +0.268  toward on track  Area_Type_Less_Developed_Village
    -0.159  toward at risk   Dependency_Ratio



In [8]:
imp = pd.concat([per_col[n].head(20).rename("mean_abs_shap").to_frame().assign(model=n) for n in SETS])
imp.index.name = "feature"
imp.reset_index()[["model", "feature", "mean_abs_shap"]].to_csv(C.RESULTS / "shap_importance.csv", index=False)
write_result("explain", {
    name: {
        "families": family_tbl[name].reset_index(),
        "permutation": perm_tbl[name].reset_index(),
        "interaction_totals": inter[name][0],
        "top_interactions": inter[name][1].head(10),
        "top_columns": per_col[name].head(15),
    } for name in SETS
} | {"coefficient_stability": stab.reset_index(names="share"), "groceries_shap_vs_implied_spend_rho": rho})
print("wrote results/shap_importance.csv, results/explain.json, shap_*.png")

wrote results/shap_importance.csv, results/explain.json, shap_*.png
